# Regulatory regions of the DNA mycoviruses

**Companion to the *Trichoderma* promoter-extraction notebook.**

### Read this before running anything

You asked for "annotated promoter sequences." Two problems, both fatal if ignored:

**1. These genomes have no annotated promoters.** GenBank records for CRESS-DNA viruses carry
`CDS`, sometimes `rep_origin` or `misc_feature`, and essentially never a `promoter` or
`regulatory` feature. Section 5 tests this on the actual fetched records rather than assuming
it — if a real promoter annotation exists anywhere, it gets reported and extracted.

**2. The `-800..-1` window from the *Trichoderma* work is meaningless here.** These genomes are
1.6-3.2 kb:

| Virus | Genome | 800 bp = | Two genes x 800 bp |
|---|---|---|---|
| SsHADV-1 | 2,166 nt | 37% | 74% of the genome |
| FgGMTV1 DNA-A | ~1.6 kb | 50% | 100% |
| SlaGemV-1 | ~2 kb | 40% | 80% |
| DsCDV1 | 3,185 nt | 25% | 50% |

The windows would overlap each other and the coding sequences. You would be extracting the whole
genome twice and calling it a promoter set.

### What this extracts instead

*Genomoviridae* have geminivirus-like architecture: **ambisense genomes with genes on both
strands, separated by intergenic regions (IRs)**. Transcription is host RNA-Pol-II driven from
promoters inside those IRs. SsHADV-1 has two ORFs — CP in positive sense, Rep in negative sense —
and two intergenic regions, one containing the replication origin. **The IRs are the
promoter-containing regions.**

### The distinction that decides whether the output is usable

Not every IR is a promoter. For a gap between upstream feature A and downstream feature B:

| Layout | Orientation | What the gap actually is |
|---|---|---|
| `A(-) ... B(+)` | **divergent** — 5' ends face the gap | **bidirectional promoter** |
| `A(+) ... B(+)` | tandem | promoter for B only |
| `A(-) ... B(-)` | tandem | promoter for A only |
| `A(+) ... B(-)` | **convergent** — 3' ends face the gap | **terminator / polyA — NOT a promoter** |

On a two-gene ambisense genome, treating convergent IRs as promoters makes **half the output
junk**. This is the main way to get this analysis wrong.

### Run order

Every cell, top to bottom. Section 7 is a self-test with hand-computed answers; it halts the
notebook if the circular-wrap or orientation logic regresses.

---
## 1. Setup

Biopython handles GenBank parsing — hand-rolling `complement(join(...))` is not worth the risk.

In [1]:
try:
    import Bio
    print("biopython", Bio.__version__)
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=False)
    import Bio; print("biopython", Bio.__version__, "(installed)")

from Bio import Entrez, SeqIO

# NCBI requires an identifying email for Entrez; requests without one may be throttled.
Entrez.email = "julia.tondera@trinity.ox.ac.uk"      # <-- CHANGE THIS
# Entrez.api_key = "..."              # optional, raises the rate limit to 10 req/s
print("Entrez.email:", Entrez.email)
if Entrez.email == "you@example.com":
    print("  WARNING: set a real address before running section 4.")

biopython 1.88 (installed)
Entrez.email: julia.tondera@trinity.ox.ac.uk


---
## 2. Targets

Only **one** accession below was verified directly: `GQ365709` for SsHADV-1 (2,166 bp, circular,
`Sclerotinia gemycircularvirus 1, complete genome`). The rest are resolved by **searching NCBI at
runtime** rather than hard-coded, because I could not confirm them and inventing accession numbers
would be worse than looking them up.

If a search returns nothing, that is reported — not silently skipped.

In [2]:
# name -> (search query, verified accessions or None)
TARGETS = {
    "SsHADV-1 (Sclerotinia gemycircularvirus 1)": (
        "Sclerotinia gemycircularvirus 1[Organism] OR Sclerotinia sclerotiorum "
        "hypovirulence-associated DNA virus 1[All Fields]",
        ["GQ365709"],                      # verified
    ),
    "FgGMTV1 (Fusarium graminearum gemytripvirus 1)": (
        "Fusarium graminearum gemytripvirus 1[All Fields]", None),
    "SlaGemV-1 (soybean leaf-associated gemygorvirus-1)": (
        "soybean leaf associated gemygorvirus 1[All Fields]", None),
    "BcGDaV1 (Botrytis cinerea gemydayirivirus 1)": (
        "Botrytis cinerea gemydayirivirus 1[All Fields]", None),
    "BcssDV1 (Botrytis cinerea ssDNA virus 1)": (
        "Botrytis cinerea ssDNA virus 1[All Fields]", None),
    "DsCDV1 (Diaporthe sojae circular DNA virus 1)": (
        "Diaporthe sojae circular DNA virus 1[All Fields]", None),
}

# FgGMTV1 is tripartite (DNA-A/B/C) and BcssDV1 tetra-segmented, so several of
# these resolve to MULTIPLE accessions. Each segment is its own circular molecule.
for k in TARGETS:
    print(" ", k)

  SsHADV-1 (Sclerotinia gemycircularvirus 1)
  FgGMTV1 (Fusarium graminearum gemytripvirus 1)
  SlaGemV-1 (soybean leaf-associated gemygorvirus-1)
  BcGDaV1 (Botrytis cinerea gemydayirivirus 1)
  BcssDV1 (Botrytis cinerea ssDNA virus 1)
  DsCDV1 (Diaporthe sojae circular DNA virus 1)


---
## 3. Resolve accessions

Each query runs against `nuccore`; hits are printed with title and length so you can see what was actually matched before anything downstream uses it.

In [3]:
import time

def resolve(query, retmax=20):
    h = Entrez.esearch(db="nuccore", term=query, retmax=retmax)
    ids = Entrez.read(h)["IdList"]; h.close()
    if not ids:
        return []
    time.sleep(0.4)
    h = Entrez.esummary(db="nuccore", id=",".join(ids))
    summ = Entrez.read(h); h.close()
    return [(s.get("AccessionVersion", s.get("Caption", "?")),
             s.get("Title", ""), int(s.get("Length", 0))) for s in summ]

RESOLVED = {}
for name, (query, known) in TARGETS.items():
    print("\n%s" % name)
    if known:
        RESOLVED[name] = list(known)
        print("   using verified accession(s):", known)
        continue
    try:
        hits = resolve(query)
    except Exception as e:
        print("   search failed: %s: %s" % (type(e).__name__, str(e)[:150])); hits = []
    if not hits:
        print("   NO HITS - excluded (reported, not silently skipped)")
        RESOLVED[name] = []
        continue
    for acc, title, ln in hits[:8]:
        print("   %-14s %5d bp  %s" % (acc, ln, title[:78]))
    keep = [a for a, t, l in hits
            if ("complete" in t.lower() or "segment" in t.lower() or "DNA-" in t) and l > 500]
    RESOLVED[name] = keep or [hits[0][0]]
    print("   -> keeping:", RESOLVED[name])
    time.sleep(0.4)

print("\n\nresolved %d/%d targets" % (sum(1 for v in RESOLVED.values() if v), len(TARGETS)))


SsHADV-1 (Sclerotinia gemycircularvirus 1)
   using verified accession(s): ['GQ365709']

FgGMTV1 (Fusarium graminearum gemytripvirus 1)
   NC_078279.1     1320 bp  Fusarium graminearum gemytripvirus 1 segment DNA-B coat protein (QK657_sBgp1) 
   NC_078278.1     1309 bp  Fusarium graminearum gemytripvirus 1 segment DNA-C hypothetical protein (QK657
   NC_078277.1     1316 bp  Fusarium graminearum gemytripvirus 1 segment DNA-A replication initiation prot
   MK430077.2      1320 bp  Fusarium graminearum gemytripvirus 1 segment DNA-B coat protein gene, complete
   MK430078.1      1309 bp  Fusarium graminearum gemytripvirus 1 segment DNA-C hypothetical protein gene, 
   MK430076.1      1316 bp  Fusarium graminearum gemytripvirus 1 segment DNA-A replication initiation prot
   -> keeping: ['NC_078279.1', 'NC_078278.1', 'NC_078277.1', 'MK430077.2', 'MK430078.1', 'MK430076.1']

SlaGemV-1 (soybean leaf-associated gemygorvirus-1)
   NO HITS - excluded (reported, not silently skipped)

BcGDaV1 (B

---
## 4. Fetch GenBank records

In [4]:
RECORDS = {}          # name -> [SeqRecord, ...]
for name, accs in RESOLVED.items():
    RECORDS[name] = []
    for acc in accs:
        try:
            h = Entrez.efetch(db="nuccore", id=acc, rettype="gbwithparts", retmode="text")
            rec = SeqIO.read(h, "genbank"); h.close()
            RECORDS[name].append(rec)
            print("%-46s %-12s %5d bp  %s"
                  % (name[:46], rec.id, len(rec.seq),
                     rec.annotations.get("topology", "unknown")))
        except Exception as e:
            print("%-46s %-12s FETCH FAILED: %s" % (name[:46], acc, str(e)[:90]))
        time.sleep(0.4)

n = sum(len(v) for v in RECORDS.values())
print("\n%d records fetched" % n)
if n == 0:
    print("Nothing fetched. Check Entrez.email and your network - sections 8+ need these.")

SsHADV-1 (Sclerotinia gemycircularvirus 1)     GQ365709.1    2166 bp  circular
FgGMTV1 (Fusarium graminearum gemytripvirus 1) NC_078279.1   1320 bp  linear
FgGMTV1 (Fusarium graminearum gemytripvirus 1) NC_078278.1   1309 bp  linear
FgGMTV1 (Fusarium graminearum gemytripvirus 1) NC_078277.1   1316 bp  linear
FgGMTV1 (Fusarium graminearum gemytripvirus 1) MK430077.2    1320 bp  linear
FgGMTV1 (Fusarium graminearum gemytripvirus 1) MK430078.1    1309 bp  linear
FgGMTV1 (Fusarium graminearum gemytripvirus 1) MK430076.1    1316 bp  linear
BcssDV1 (Botrytis cinerea ssDNA virus 1)       OR146577.1    1707 bp  circular
BcssDV1 (Botrytis cinerea ssDNA virus 1)       OR146576.1    1062 bp  linear
BcssDV1 (Botrytis cinerea ssDNA virus 1)       OR146575.1    1076 bp  linear
BcssDV1 (Botrytis cinerea ssDNA virus 1)       OR146574.1    1708 bp  circular
BcssDV1 (Botrytis cinerea ssDNA virus 1)       OR146573.1    1672 bp  linear
BcssDV1 (Botrytis cinerea ssDNA virus 1)       OR146572.1    1143 bp  

---
## 5. What is *actually* annotated

The empirical answer to "are there annotated promoters?" Do not take my word for it — this cell
counts every feature type present across all fetched records.

In [5]:
from collections import Counter

types = Counter()
for name, recs in RECORDS.items():
    for r in recs:
        for f in r.features:
            types[f.type] += 1

print("feature types across all records:")
for t, c in types.most_common():
    print("   %-18s %d" % (t, c))

REGULATORY = {"promoter", "regulatory", "TATA_signal", "minus_35_signal",
              "minus_10_signal", "enhancer", "CAAT_signal", "GC_signal"}
found = {t: c for t, c in types.items() if t in REGULATORY}
print("\ntrue promoter/regulatory feature types found:", found or "NONE")
if not found:
    print("  -> Confirms the premise: there are no annotated promoters to isolate.")
    print("     The intergenic regions below are the correct substitute.")
else:
    print("  -> Some exist. Section 9 exports them alongside the intergenic regions.")

feature types across all records:
   CDS                30
   source             28
   gene               5
   misc_feature       2

true promoter/regulatory feature types found: NONE
  -> Confirms the premise: there are no annotated promoters to isolate.
     The intergenic regions below are the correct substitute.


---
## 6. Extraction code

### 6.1 Feature model and orientation logic

In [6]:
import re
from dataclasses import dataclass, field

CRESS_NONANUCLEOTIDES = {
    "TAATATTAT": "canonical geminivirus/genomovirus",
    "TAGTATTAC": "circovirus-like variant",
    "TATTATTAT": "variant",
    "AAGTATTAC": "nanovirus-like variant",
}
_COMP = str.maketrans("ACGTRYSWKMBDHVNacgtryswkmbdhvn", "TGCAYRSWMKVHDBNtgcayrswmkvhdbn")

def revcomp(s):
    return s.translate(_COMP)[::-1]

@dataclass
class Feat:
    name: str; start: int; end: int; strand: int; kind: str = "CDS"   # 0-based half-open

@dataclass
class Region:
    accession: str; index: int; start: int; end: int; length: int
    wraps_origin: bool; upstream_feature: str; downstream_feature: str
    orientation: str; is_promoter: bool; promotes: tuple; sequence: str
    oriented_sequences: dict = field(default_factory=dict)
    ori_motifs: list = field(default_factory=list)

def normalise_features(feats, seq_len, merge_overlaps=True):
    # Compact viral genomes have overlapping ORFs; without merging they would
    # produce negative-length gaps.
    fs = sorted([f for f in feats if f.end > f.start], key=lambda f: (f.start, f.end))
    if not merge_overlaps:
        return fs
    merged = []
    for f in fs:
        if merged and f.start < merged[-1].end:
            p = merged[-1]
            merged[-1] = Feat(p.name + "|" + f.name, p.start, max(p.end, f.end),
                              p.strand if p.strand == f.strand else 0, "merged")
        else:
            merged.append(f)
    return merged

def classify_gap(up, down):
    # Returns (orientation, is_promoter, genes_promoted). Strand 0 = merged mixed block.
    a, b = up.strand, down.strand
    if a == 0 or b == 0:  return "ambiguous", False, ()
    if a < 0 and b > 0:   return "divergent", True, (up.name, down.name)
    if a > 0 and b < 0:   return "convergent", False, ()
    if a > 0 and b > 0:   return "tandem_forward", True, (down.name,)
    return "tandem_reverse", True, (up.name,)

def find_ori_motifs(seq, also_revcomp=True):
    hits = []
    for motif, desc in CRESS_NONANUCLEOTIDES.items():
        for m in re.finditer("(?=(%s))" % motif, seq.upper()):
            hits.append((motif, m.start(), "+", desc))
        if also_revcomp:
            for m in re.finditer("(?=(%s))" % revcomp(motif), seq.upper()):
                hits.append((motif, m.start(), "-", desc))
    return sorted(hits, key=lambda h: h[1])

### 6.2 Circular-aware intergenic extraction

The wrap-around region — from the end of the last feature, over the origin, to the start of the
first — is the case most likely to be silently mishandled. It is built explicitly and checked
against a hand-computed answer in section 7.

Oriented output follows the same convention as the *Trichoderma* notebook: the returned sequence
reads 5'->3' **for the gene it drives**, so its last base is the one immediately 5' of that
gene's start. Minus-strand genes get the reverse complement. A divergent IR therefore yields two
sequences from the same interval, one per direction.

In [7]:
def extract_intergenic(accession, seq, feats, circular=True, min_length=1,
                       max_promoter_window=None):
    L = len(seq)
    fs = normalise_features(feats, L)
    if not fs:
        return []

    gaps = [(fs[i], fs[i+1], fs[i].end, fs[i+1].start, False) for i in range(len(fs)-1)]
    if circular:
        gaps.append((fs[-1], fs[0], fs[-1].end, fs[0].start, True))   # over the origin
    else:
        if fs[0].start > 0:
            gaps.insert(0, (None, fs[0], 0, fs[0].start, False))
        if fs[-1].end < L:
            gaps.append((fs[-1], None, fs[-1].end, L, False))

    out, idx = [], 0
    for up, down, gs, ge, wraps in gaps:
        if wraps:
            length, sub = (L - gs) + ge, seq[gs:] + seq[:ge]
        else:
            length, sub = ge - gs, seq[gs:ge]
        if length < min_length:
            continue

        if up is None or down is None:
            orient, is_prom = "terminal", down is not None
            promotes = (down.name,) if (down is not None and down.strand > 0) else ()
        else:
            orient, is_prom, promotes = classify_gap(up, down)

        oriented = {}
        for gene in promotes:
            gene_strand = down.strand if (down and gene == down.name) else (up.strand if up else 1)
            s = sub if gene_strand > 0 else revcomp(sub)
            if max_promoter_window:
                s = s[-max_promoter_window:]      # keep the end proximal to the gene
            oriented[gene] = s

        out.append(Region(accession, idx, gs, ge, length, wraps,
                          up.name if up else "-", down.name if down else "-",
                          orient, is_prom, tuple(promotes), sub, oriented,
                          find_ori_motifs(sub)))
        idx += 1
    return out

def features_from_biopython(record, kinds=("CDS", "gene")):
    # Prefer CDS; keep gene rows only where no CDS overlaps, so records
    # annotating both do not yield duplicates.
    out, cds = [], []
    for f in record.features:
        if f.type not in kinds:
            continue
        name = (f.qualifiers.get("gene") or f.qualifiers.get("product")
                or f.qualifiers.get("locus_tag") or [f.type])[0]
        fe = Feat(name, int(f.location.start), int(f.location.end),
                  1 if f.location.strand in (1, None) else -1, f.type)
        (cds if f.type == "CDS" else out).append(fe)
    if cds:
        return cds + [g for g in out
                      if not any(g.start < c.end and c.start < g.end for c in cds)]
    return out

def annotated_regulatory_features(record):
    wanted = {"promoter", "regulatory", "TATA_signal", "misc_signal",
              "protein_bind", "rep_origin", "misc_feature"}
    return [{"type": f.type, "start": int(f.location.start), "end": int(f.location.end),
             "strand": f.location.strand,
             "regulatory_class": (f.qualifiers.get("regulatory_class") or [""])[0],
             "note": (f.qualifiers.get("note") or [""])[0]}
            for f in record.features if f.type in wanted]

---
## 7. Self-test — hand-computed, halts on failure

A synthetic 100 nt circular genome laid out like SsHADV-1: CP on the plus strand at 10-30, Rep on
the minus strand at 50-70. The correct answer, worked out by hand:

- gap **30..50** sits between CP(+) and Rep(-) → **convergent** → terminator, not a promoter
- gap **70..10** wraps the origin, Rep(-) then CP(+) → **divergent** → bidirectional promoter,
  length 30 + 10 = **40**

In [8]:
GEN = "".join("ACGT"[(i // 7) % 4] for i in range(100))
_ok = _fail = 0
def _ck(n, c, x=""):
    global _ok, _fail
    if c: _ok += 1; print("  PASS %-50s %s" % (n, x))
    else: _fail += 1; print("  FAIL %-50s %s" % (n, x))

P = lambda s: Feat("x", 0, 1, s)
_ck("A(-) B(+) -> divergent, promoter",      classify_gap(P(-1), P(1))[:2] == ("divergent", True))
_ck("A(+) B(-) -> convergent, NOT promoter", classify_gap(P(1), P(-1))[:2] == ("convergent", False))
_ck("A(+) B(+) -> tandem_forward",           classify_gap(P(1), P(1))[:2] == ("tandem_forward", True))
_ck("A(-) B(-) -> tandem_reverse",           classify_gap(P(-1), P(-1))[:2] == ("tandem_reverse", True))
_ck("mixed merged block -> ambiguous",       classify_gap(Feat("m",0,1,0), P(1))[:2] == ("ambiguous", False))

feats = [Feat("cp", 10, 30, +1), Feat("rep", 50, 70, -1)]
regs = extract_intergenic("TEST", GEN, feats, circular=True)
_ck("two intergenic regions", len(regs) == 2, str(len(regs)))
conv = [r for r in regs if r.orientation == "convergent"][0]
divg = [r for r in regs if r.orientation == "divergent"][0]

_ck("convergent IR not a promoter",      conv.is_promoter is False)
_ck("convergent IR promotes nothing",    conv.promotes == ())
_ck("convergent IR coords 30..50",       (conv.start, conv.end) == (30, 50))
_ck("convergent seq == GEN[30:50]",      conv.sequence == GEN[30:50])

_ck("divergent IR wraps the origin",     divg.wraps_origin is True)
_ck("divergent IR length 30+10 = 40",    divg.length == 40, str(divg.length))
_ck("wrapped seq == GEN[70:] + GEN[:10]", divg.sequence == GEN[70:] + GEN[:10])
_ck("seq length matches reported length", len(divg.sequence) == divg.length)
_ck("divergent promotes BOTH genes",     set(divg.promotes) == {"rep", "cp"}, str(divg.promotes))

_ck("cp promoter plus-strand as-is",     divg.oriented_sequences["cp"] == divg.sequence)
_ck("cp promoter ends at GEN[9]",        divg.oriented_sequences["cp"][-1] == GEN[9])
_ck("rep promoter reverse-complemented", divg.oriented_sequences["rep"] == revcomp(divg.sequence))
_ck("rep promoter ends at complement of GEN[70]",
    divg.oriented_sequences["rep"][-1] == revcomp(GEN[70]))
_ck("convergent IR yields no oriented seqs", conv.oriented_sequences == {})

w = [r for r in extract_intergenic("T", GEN, feats, True, max_promoter_window=15)
     if r.orientation == "divergent"][0]
_ck("window truncates to 15",            len(w.oriented_sequences["cp"]) == 15)
_ck("truncation keeps gene-proximal end", w.oriented_sequences["cp"] == w.sequence[-15:])

_ck("overlapping ORFs merged, no negative gap",
    all(r.length > 0 for r in extract_intergenic("T", GEN,
        [Feat("a",10,40,1), Feat("b",30,60,1)], True)))
one = extract_intergenic("T", GEN, [Feat("rep", 20, 50, -1)], True)
_ck("single gene -> one wrapping IR of length 70", len(one) == 1 and one[0].length == 70)
_ck("ori nonanucleotide found at right offset",
    any(x[0] == "TAATATTAT" and x[1] == 4
        for x in find_ori_motifs("GGGG" + "TAATATTAT" + "CCCC")))
_ck("no ori false positives in ACGT repeat", find_ori_motifs("ACGT" * 20) == [])

print("\n" + "=" * 60)
print("SELF-TEST  PASSED %d  FAILED %d" % (_ok, _fail))
print("=" * 60)
assert _fail == 0, "Self-test failed - do NOT trust the output below."

  PASS A(-) B(+) -> divergent, promoter                   
  PASS A(+) B(-) -> convergent, NOT promoter              
  PASS A(+) B(+) -> tandem_forward                        
  PASS A(-) B(-) -> tandem_reverse                        
  PASS mixed merged block -> ambiguous                    
  PASS two intergenic regions                             2
  PASS convergent IR not a promoter                       
  PASS convergent IR promotes nothing                     
  PASS convergent IR coords 30..50                        
  PASS convergent seq == GEN[30:50]                       
  PASS divergent IR wraps the origin                      
  PASS divergent IR length 30+10 = 40                     40
  PASS wrapped seq == GEN[70:] + GEN[:10]                 
  PASS seq length matches reported length                 
  PASS divergent promotes BOTH genes                      ('rep', 'cp')
  PASS cp promoter plus-strand as-is                      
  PASS cp promoter ends at GEN[9]       

---
## 8. Run on the real genomes

In [9]:
ALL_REGIONS = []      # (virus_name, record, Region)

for name, recs in RECORDS.items():
    if not recs:
        continue
    print("\n" + "=" * 78)
    print(name)
    print("=" * 78)
    for rec in recs:
        feats = features_from_biopython(rec)
        topo = rec.annotations.get("topology", "").lower()
        circular = (topo == "circular") or not topo    # CRESS genomes are circular
        if topo and topo != "circular":
            print("  NOTE: topology='%s' - treating as circular anyway (CRESS-DNA)." % topo)
        regs = extract_intergenic(rec.id, str(rec.seq), feats, circular=circular)
        print("\n  %s  %d bp  |  %d coding features  |  %d intergenic regions"
              % (rec.id, len(rec.seq), len(feats), len(regs)))
        for f in sorted(feats, key=lambda x: x.start):
            print("     CDS %-22s %5d-%-5d strand %+d" % (f.name[:22], f.start, f.end, f.strand))
        for r in regs:
            tag = "PROMOTER" if r.is_promoter else "not a promoter"
            wrap = " [wraps origin]" if r.wraps_origin else ""
            print("     IR%d  %5d-%-5d  %4d bp  %-15s %-14s %s%s"
                  % (r.index, r.start, r.end, r.length, r.orientation, tag,
                     "->" + ",".join(r.promotes) if r.promotes else "", wrap))
            if r.ori_motifs:
                print("            ori motif(s):",
                      ", ".join("%s@%d(%s)" % (m, p, s) for m, p, s, _ in r.ori_motifs))
            ALL_REGIONS.append((name, rec, r))

print("\n\ntotal intergenic regions: %d  |  promoter-bearing: %d"
      % (len(ALL_REGIONS), sum(1 for _, _, r in ALL_REGIONS if r.is_promoter)))


SsHADV-1 (Sclerotinia gemycircularvirus 1)

  GQ365709.1  2166 bp  |  2 coding features  |  2 intergenic regions
     CDS V1                        95-1034  strand +1
     CDS AC1                     1153-2128  strand -1
     IR0   1034-1153    119 bp  convergent      not a promoter 
     IR1   2128-95      133 bp  divergent       PROMOTER       ->AC1,V1 [wraps origin]
            ori motif(s): TAATATTAT@38(+)

FgGMTV1 (Fusarium graminearum gemytripvirus 1)
  NOTE: topology='linear' - treating as circular anyway (CRESS-DNA).

  NC_078279.1  1320 bp  |  1 coding features  |  2 intergenic regions
     CDS CP                       141-1044  strand +1
     IR0      0-141     141 bp  terminal        PROMOTER       ->CP
     IR1   1044-1320    276 bp  terminal        not a promoter 
  NOTE: topology='linear' - treating as circular anyway (CRESS-DNA).

  NC_078278.1  1309 bp  |  1 coding features  |  2 intergenic regions
     CDS hypothetical protein     205-892   strand -1
     IR0      0-2

---
## 9. Export

One row per **(intergenic region, gene it drives)**, so a divergent IR produces two rows with the
same interval in opposite orientations. Convergent IRs are included but flagged
`is_promoter=False` — kept so you can see what was excluded rather than having it vanish.

In [10]:
import csv

COLS = ["virus", "accession", "segment_length", "region_index", "orientation",
        "is_promoter", "promotes_gene", "gene_strand", "region_start", "region_end",
        "region_length", "wraps_origin", "upstream_feature", "downstream_feature",
        "ori_motifs", "gc_percent", "sequence"]

rows = []
for name, rec, r in ALL_REGIONS:
    ori = ";".join("%s@%d(%s)" % (m, p, s) for m, p, s, _ in r.ori_motifs)
    targets = r.promotes if r.promotes else [None]
    for gene in targets:
        seq = r.oriented_sequences.get(gene, r.sequence)
        gc = sum(1 for c in seq.upper() if c in "GC")
        rows.append({
            "virus": name, "accession": rec.id, "segment_length": len(rec.seq),
            "region_index": r.index, "orientation": r.orientation,
            "is_promoter": r.is_promoter, "promotes_gene": gene or "",
            "gene_strand": (("+" if seq == r.sequence else "-") if gene else ""),
            "region_start": r.start, "region_end": r.end, "region_length": r.length,
            "wraps_origin": r.wraps_origin,
            "upstream_feature": r.upstream_feature,
            "downstream_feature": r.downstream_feature,
            "ori_motifs": ori,
            "gc_percent": round(100.0 * gc / max(len(seq), 1), 2),
            "sequence": seq,
        })

with open("mycovirus_regulatory_regions.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=COLS, delimiter=";")
    w.writeheader(); w.writerows(rows)
print("wrote %d rows -> mycovirus_regulatory_regions.csv" % len(rows))

with open("mycovirus_promoters.fasta", "w") as fh:
    n_fa = 0
    for r_ in rows:
        if not r_["is_promoter"]:
            continue
        fh.write(">%s|%s|IR%s|%s|%s\n%s\n" % (
            r_["virus"].split(" (")[0], r_["accession"], r_["region_index"],
            r_["orientation"], r_["promotes_gene"], r_["sequence"]))
        n_fa += 1
print("promoter-only FASTA written for %d sequences" % n_fa)

wrote 53 rows -> mycovirus_regulatory_regions.csv
promoter-only FASTA written for 30 sequences


---
## 10. Limitations — say these out loud before anyone asks

1. **These are not mapped promoters.** They are intergenic regions that *contain* the promoters.
   Locating actual transcription start sites needs 5' RACE or CAGE data, which for these viruses
   largely does not exist.
2. **Orientation is inferred from gene layout, not from transcription data.** The
   divergent/convergent logic is a strong structural prior, not a measurement.
3. **`min_length=1` keeps very short IRs.** In compact genomes some gaps are a few bp and are not
   plausible promoters. Filter on `region_length` before using the output.
4. **Overlapping ORFs are merged into one block** with strand `0`; gaps adjacent to such a block
   are marked `ambiguous` rather than guessed at.
5. **The nonanucleotide search double-reports near-palindromes.** `TAATATTAT` and its reverse
   complement `ATAATATTA` share 8 of 9 bases, so one true site in the right context appears as two
   hits one position apart. Treat adjacent +/- pairs as a single site.
6. **Segment counts vary.** FgGMTV1 is tripartite and BcssDV1 tetra-segmented; each segment is
   processed as an independent circular molecule, so per-virus row counts differ a lot.
7. **Every accession except `GQ365709` is resolved by text search at runtime.** Check section 3's
   output actually matched the virus you meant before trusting section 8.